In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "8"
os.environ["OPENBLAS_NUM_THREADS"] = "8"
os.environ["MKL_NUM_THREADS"] = "8"
os.environ["VECLIB_MAXIMUM_THREADS"] = "8"
os.environ["NUMEXPR_NUM_THREADS"] = "8"

In [2]:
import pandas as pd
import numpy as np

DATA_DIR = '../data/'

train = pd.read_csv(DATA_DIR + 'GLC25_PA_metadata_train.csv')
test  = pd.read_csv(DATA_DIR + 'GLC25_PA_metadata_test.csv')

print('Train shape:', train.shape)
print('Test  shape:', test.shape)

Train shape: (1483637, 9)
Test  shape: (14784, 8)


In [3]:
print(train.dtypes)
print(test.dtypes)

lon                  float64
lat                  float64
year                   int64
geoUncertaintyInM    float64
areaInM2             float64
region                object
country               object
speciesId            float64
surveyId               int64
dtype: object
lon                  float64
lat                  float64
year                   int64
geoUncertaintyInM    float64
areaInM2             float64
region                object
country               object
surveyId               int64
dtype: object


In [4]:
print('=== Valeurs manquantes TRAIN ===')
print(train.isnull().sum())
print('\n=== Valeurs manquantes TEST ===')
print(test.isnull().sum())

=== Valeurs manquantes TRAIN ===
lon                       0
lat                       0
year                      0
geoUncertaintyInM     12496
areaInM2             183272
region                    0
country                   0
speciesId                 0
surveyId                  0
dtype: int64

=== Valeurs manquantes TEST ===
lon                    0
lat                    0
year                   0
geoUncertaintyInM    840
areaInM2             644
region                 0
country                0
surveyId               0
dtype: int64


In [5]:
train['lat_lon'] = train['lat'] * train['lon']
train['lat2']= train['lat'] ** 2
train['lon2'] = train['lon'] ** 2

In [6]:
print(train.corr(numeric_only=True)["speciesId"].sort_values(ascending=False))

speciesId            1.000000
lat2                 0.015254
lat                  0.014907
lat_lon              0.005264
year                 0.003626
lon                  0.003009
geoUncertaintyInM    0.001994
surveyId             0.000730
lon2                 0.000599
areaInM2            -0.017997
Name: speciesId, dtype: float64


1. Préparation des données

In [7]:
features = ['lon', 'lat','lat_lon', 'lat2', 'year']

X = train.groupby('surveyId')[features + ['region', 'country']].first()
X = pd.get_dummies(X, columns=['region', 'country'])


In [8]:
TOP_K_SPECIES = 350

top_species = train['speciesId'].value_counts().head(TOP_K_SPECIES).index

train_filtered = train[train['speciesId'].isin(top_species)]

Y = (
    train_filtered
    .assign(value=1)
    .pivot_table(index='surveyId', columns='speciesId', values='value', fill_value=0)
)

X = X.loc[Y.index]

In [9]:
from xgboost import XGBClassifier
import numpy as np

# nettoyage
X = X.astype(float)
X = X.replace([np.inf, -np.inf], 0)
X = X.fillna(0)

models = {}

for species in Y.columns:
    y = Y[species].astype(int)
    
    # skip espèces inutiles
    if y.nunique() < 2:
        continue
    
    print(f"Training species {species}")
    
    model = XGBClassifier(
        n_estimators=200,   # ↓ pour commencer (rapide)
        max_depth=5,
        learning_rate=0.1,
        tree_method='hist',
        n_jobs=8
    )
    
    model.fit(X, y)
    models[species] = model

Training species 53.0
Training species 96.0
Training species 129.0
Training species 146.0
Training species 167.0
Training species 249.0
Training species 254.0
Training species 262.0
Training species 300.0
Training species 340.0
Training species 391.0
Training species 394.0
Training species 423.0
Training species 433.0
Training species 469.0
Training species 478.0
Training species 540.0
Training species 544.0
Training species 559.0
Training species 581.0
Training species 593.0
Training species 623.0
Training species 643.0
Training species 651.0
Training species 661.0
Training species 694.0
Training species 791.0
Training species 799.0
Training species 822.0
Training species 843.0
Training species 868.0
Training species 890.0
Training species 907.0
Training species 958.0
Training species 963.0
Training species 976.0
Training species 981.0
Training species 1015.0
Training species 1018.0
Training species 1037.0
Training species 1051.0
Training species 1085.0
Training species 1086.0
Trainin

TEST 

In [10]:
test['lon_lat'] = test['lon'] * test['lat']
test['lat2']= test['lat'] ** 2

In [11]:
features = ['lon', 'lat', 'lon_lat', 'year', 'lat2']

X_test = test.groupby('surveyId')[features + ['region', 'country']].first()
X_test = pd.get_dummies(X_test, columns=['region', 'country'])

# aligner avec train
X_test = X_test.reindex(columns=X.columns, fill_value=0)

# nettoyage
import numpy as np
X_test = X_test.astype(float)
X_test = X_test.replace([np.inf, -np.inf], 0)
X_test = X_test.fillna(0)

In [12]:
preds = {}

for species, model in models.items():
    preds[species] = model.predict_proba(X_test)[:, 1]

preds = pd.DataFrame(preds, index=X_test.index)

In [13]:
preds.head(5)

,53.0,96.0,129.0,146.0,167.0,249.0,254.0,262.0,300.0,340.0,...,10950.0,10969.0,10998.0,11005.0,11054.0,11078.0,11140.0,11157.0,11193.0,11195.0
surveyId,,,,,,,,,,,,,,,,,,,,,
642,0.005488,0.012598,0.038179,0.000053,0.002017,0.034465,0.116547,0.000164,0.000379,0.203698,...,0.016987,0.129891,0.002160,0.016954,0.001533,0.060916,0.105320,0.000003,0.000025,0.384441
1792,0.014518,0.006064,0.000584,0.106708,0.002686,0.016033,0.030029,0.218116,0.000235,0.011779,...,0.006030,0.003329,0.003988,0.150007,0.012557,0.000794,0.119027,0.002009,0.000025,0.068628
3256,0.004346,0.054685,0.043993,0.024426,0.005635,0.073767,0.100929,0.000066,0.000060,0.017378,...,0.006487,0.033135,0.000734,0.001821,0.007880,0.000065,0.032372,0.000659,0.000095,0.033383
3855,0.114316,0.035885,0.000248,0.059931,0.007307,0.055709,0.004869,0.002109,0.000237,0.041593,...,0.000282,0.006644,0.016643,0.084206,0.007250,0.000823,0.063592,0.002030,0.000012,0.067667
4889,0.073298,0.016027,0.078910,0.000517,0.221058,0.096970,0.393432,0.000352,0.001368,0.122572,...,0.016677,0.131022,0.000089,0.019314,0.000242,0.023540,0.079876,0.000005,0.000017,0.417488


In [14]:
K = 25  # valeur recommandée

def get_top_k(row):
    return list(row.sort_values(ascending=False).head(K).index)

predictions = preds.apply(get_top_k, axis=1)

In [15]:
submission = pd.DataFrame({
    'surveyId': predictions.index,
    'predictions': predictions.apply(lambda x: ' '.join(map(str, x)))
})

print(submission)

          surveyId                                        predictions
surveyId                                                             
642            642  6746.0 4397.0 9465.0 2556.0 9024.0 8431.0 6310...
1792          1792  5557.0 3451.0 1015.0 262.0 5543.0 9714.0 11005...
3256          3256  963.0 10600.0 1888.0 7999.0 9669.0 9647.0 8512...
3855          3855  9647.0 8818.0 10268.0 10073.0 6925.0 6883.0 14...
4889          4889  7122.0 9024.0 6746.0 8431.0 11195.0 9341.0 254...
...            ...                                                ...
5010108    5010108  10274.0 433.0 53.0 2474.0 9647.0 7880.0 6883.0...
5010109    5010109  2474.0 5868.0 11078.0 9647.0 11140.0 8705.0 23...
5010110    5010110  2474.0 5868.0 11078.0 9647.0 11140.0 8705.0 23...
5010111    5010111  2474.0 5868.0 11078.0 9647.0 11140.0 8705.0 23...
5010112    5010112  2474.0 5868.0 11078.0 9647.0 11140.0 8705.0 23...

[14784 rows x 2 columns]


In [16]:
submission.to_csv('submission.csv', index=False)

In [17]:
from sklearn.metrics import f1_score

X_train = X.loc[Y.index]
y_true = Y.astype(int)

y_proba = pd.DataFrame(
    {species: model.predict_proba(X_train)[:, 1] for species, model in models.items()},
    index=X_train.index
)

K = 18
y_pred = (y_proba.rank(axis=1, method='first', ascending=False) <= K).astype(int)
y_pred = y_pred[y_true.columns]

f1_macro = f1_score(y_true, y_pred, average='macro')
f1_micro = f1_score(y_true, y_pred, average='micro')
f1_weighted = f1_score(y_true, y_pred, average='weighted')

print("F1 macro :", f1_macro)
f1_per_species = y_true.apply(
    lambda col: f1_score(col, y_pred[col.name]),
    axis=0
)

print("F1 weighted :", f1_weighted)


F1 macro : 0.3391101328434718
F1 weighted : 0.3751240891734011


In [18]:
from sklearn.metrics import f1_score

best_k = 0
best_score = 0

for K in range(1, 51):  # tester de 1 à 50
    y_pred_k = (y_proba.rank(axis=1, method='first', ascending=False) <= K).astype(int)
    y_pred_k = y_pred_k[y_true.columns]
    
    score = f1_score(y_true, y_pred_k, average='micro')
    
    print(f"K={K} → F1={score:.4f}")
    
    if score > best_score:
        best_score = score
        best_k = K

print("\nBest K:", best_k)
print("Best F1:", best_score)

K=1 → F1=0.0903
K=2 → F1=0.1574
K=3 → F1=0.2096
K=4 → F1=0.2509
K=5 → F1=0.2841
K=6 → F1=0.3105
K=7 → F1=0.3319
K=8 → F1=0.3486
K=9 → F1=0.3621
K=10 → F1=0.3728
K=11 → F1=0.3812
K=12 → F1=0.3882
K=13 → F1=0.3932
K=14 → F1=0.3970
K=15 → F1=0.3999
K=16 → F1=0.4018
K=17 → F1=0.4026
K=18 → F1=0.4031
K=19 → F1=0.4029
K=20 → F1=0.4024
K=21 → F1=0.4015
K=22 → F1=0.4002
K=23 → F1=0.3985
K=24 → F1=0.3966
K=25 → F1=0.3944
K=26 → F1=0.3920
K=27 → F1=0.3896
K=28 → F1=0.3870
K=29 → F1=0.3843
K=30 → F1=0.3815
K=31 → F1=0.3786
K=32 → F1=0.3757
K=33 → F1=0.3727
K=34 → F1=0.3698
K=35 → F1=0.3667
K=36 → F1=0.3637
K=37 → F1=0.3606
K=38 → F1=0.3575
K=39 → F1=0.3545
K=40 → F1=0.3515
K=41 → F1=0.3485
K=42 → F1=0.3454
K=43 → F1=0.3424
K=44 → F1=0.3395
K=45 → F1=0.3365
K=46 → F1=0.3335
K=47 → F1=0.3306
K=48 → F1=0.3277
K=49 → F1=0.3249
K=50 → F1=0.3220

Best K: 18
Best F1: 0.4031051965207116


In [20]:
print(y_pred_k)

          53.0     96.0     129.0    146.0    167.0    249.0    254.0    \
surveyId                                                                  
212             0        0        0        0        0        0        0   
222             0        0        0        0        0        0        1   
243             0        0        1        0        0        0        1   
324             0        0        0        1        0        0        0   
333             0        0        0        1        0        0        0   
...           ...      ...      ...      ...      ...      ...      ...   
3919553         0        0        0        0        0        0        1   
3919592         1        0        0        0        0        0        1   
3919620         0        0        0        0        0        0        1   
3919640         1        0        0        0        0        0        1   
3919655         1        0        0        0        0        0        1   

          262.0    300.0